# 00 First Step — CIFAR-100 Baseline

This is the **baseline notebook** for CIFAR-100 image classification.

**What to do:** Just run all cells in order from top to bottom and check the output.

1. Run **Cell 1** — Mount your Google Drive and move into the project folder.
2. Run **Cell 2** — Save the training script (`00first.py`) to your Drive.
3. Run **Cell 3** — Execute the script and wait for the result.

At the end, you will see a line like:
```
Accuracy of the network on the 10000 test images: x.xxx %
```
Record this number — it is the **baseline accuracy** you will try to improve in later notebooks.

> ⚠️ Training takes a few minutes. Do not interrupt the cell while it is running.


In [4]:
# Cell 1: Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

PROJECT_PATH="/content/drive/MyDrive/Practical-ML/cifar100"
!mkdir -p "{PROJECT_PATH}"
%cd "{PROJECT_PATH}"
!ls *.py

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/drive/MyDrive/Practical-ML/cifar100
00first.py


In [ ]:
# Cell 2 : Save this cell as a Python file (Execute after editing)
%%writefile 00first.py
# Reference: https://pytorch.org/tutorials/beginner/blitz/cifar10_tutorial.html
# Adapted for CIFAR-100

# PyTorch core library for tensor operations and neural network building blocks
import torch
# nn module provides layer definitions (Conv2d, Linear, etc.) and loss functions
import torch.nn as nn
# F provides functional-style operations such as relu, softmax, etc.
import torch.nn.functional as F
# torchvision provides standard datasets (CIFAR-100) and image utilities
import torchvision
# transforms provides image preprocessing pipelines (ToTensor, Normalize, etc.)
import torchvision.transforms as transforms
# optim provides optimization algorithms such as SGD and Adam
import torch.optim as optim

# time is used to measure elapsed training time per reporting interval
import time

# Number of images processed in one forward/backward pass.
# Smaller batch_size uses less GPU memory but has noisier gradient estimates.
batch_size = 4

# Number of complete passes through the entire training dataset.
# More epochs generally improve accuracy but increase training time.
epochs = 2

# ---------------------------------------------------------------------------
# Data preparation
# ---------------------------------------------------------------------------

# Define a preprocessing pipeline applied to every image before it enters the network.
# transforms.Compose chains multiple transforms in order:
#   1. ToTensor()   : converts a PIL image (H x W x C, uint8 [0,255])
#                     to a float32 tensor (C x H x W, [0.0, 1.0])
#   2. Normalize()  : applies channel-wise normalization:
#                     output = (input - mean) / std
#                     Here mean=0.5 and std=0.5 for all 3 RGB channels,
#                     so the pixel range [0,1] is mapped to [-1, +1].
#                     Centering the data around 0 helps gradient flow during training.
transform = transforms.Compose(
    [transforms.ToTensor(),
     transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))])

# Download and load the CIFAR-100 training split.
# CIFAR-100 contains 60,000 color images of size 32x32 pixels,
# divided into 100 fine-grained classes (e.g. apple, mushroom, rocket).
# train=True  : loads the 50,000 training images
# download=True: automatically downloads the dataset if not already present
trainset = torchvision.datasets.CIFAR100(
    root='./data', train=True, download=True, transform=transform)

# DataLoader wraps the dataset and provides an iterable over mini-batches.
# batch_size : number of images per mini-batch
# shuffle=True: randomizes the order of images each epoch to reduce overfitting
# num_workers : number of parallel worker processes for loading data in the background
trainloader = torch.utils.data.DataLoader(
    trainset, batch_size=batch_size, shuffle=True, num_workers=2)

# Download and load the CIFAR-100 test split (10,000 images, 100 per class).
# train=False : loads the held-out test set used only for evaluation, not training
# shuffle=False: order does not matter for evaluation; keeps results reproducible
# batch_size=100: evaluate 100 images at a time for efficiency
testset = torchvision.datasets.CIFAR100(
    root='./data', train=False, download=True, transform=transform)
testloader = torch.utils.data.DataLoader(
    testset, batch_size=100, shuffle=False, num_workers=2)

# CIFAR-100 has 100 fine-grained classes organized into 20 superclasses
# (e.g. superclass 'flowers' contains roses, tulips, sunflowers, etc.)

# ---------------------------------------------------------------------------
# Network definition
# ---------------------------------------------------------------------------

# Define a simple CNN (Convolutional Neural Network) by subclassing nn.Module.
# All custom networks in PyTorch must inherit from nn.Module and implement
# __init__ (to define layers) and forward (to define the computation graph).
class Net(nn.Module):
    def __init__(self):
        super(Net, self).__init__()

        # --- Convolutional layers ---
        # Conv2d(in_channels, out_channels, kernel_size, padding, padding_mode)
        #
        # conv1: 3 input channels (RGB) -> 16 feature maps
        #   kernel_size=3     : 3x3 spatial filter
        #   padding=1         : adds 1 pixel border so output size = input size (32x32)
        #   padding_mode='replicate': border pixels are filled by replicating edge values,
        #                             which avoids the artificial zero-border artifact
        self.conv1 = nn.Conv2d(3, 16, 3, padding=1, padding_mode='replicate')

        # conv2: 16 feature maps -> 32 feature maps
        #   padding_mode='zeros': border pixels are filled with 0 (default behavior)
        self.conv2 = nn.Conv2d(16, 32, 3, padding=1, padding_mode='zeros')

        # --- Pooling layer ---
        # MaxPool2d(kernel_size, stride): takes the maximum value in each 2x2 window
        # and moves the window by 2 pixels (stride=2), halving the spatial dimensions.
        # e.g. 32x32 -> 16x16, 16x16 -> 8x8
        # This reduces computation and introduces spatial invariance.
        self.pool = nn.MaxPool2d(2, 2)

        # --- Flatten layer ---
        # Converts a multi-dimensional tensor (B, C, H, W) into a 1D vector per sample (B, C*H*W).
        # Required before passing the feature maps into fully connected layers.
        self.flatten = nn.Flatten()

        # --- Fully connected layers ---
        # After two conv+pool operations on 32x32 images:
        #   32x32 -> (pool) 16x16 -> (pool) 8x8
        # with 32 feature maps, the flattened size is 32 * 8 * 8 = 2048.
        #
        # fc1: 2048 -> 128  (compresses spatial features into a dense representation)
        self.fc1 = nn.Linear(32 * 8 * 8, 128)
        # fc2: 128 -> 64    (further compression)
        self.fc2 = nn.Linear(128, 64)
        # fc3: 64 -> 100    (output: one logit score per CIFAR-100 class)
        self.fc3 = nn.Linear(64, 100)

    def forward(self, x):
        # --- Block 1: conv1 + ReLU + MaxPool ---
        # Input shape : (B, 3, 32, 32)  B = batch size
        x = self.conv1(x)   # (B, 3, 32, 32) -> (B, 16, 32, 32)
        # ReLU (Rectified Linear Unit): replaces all negative values with 0.
        # Introduces non-linearity, enabling the network to learn complex patterns.
        # Formula: relu(x) = max(0, x)
        x = F.relu(x)       # (B, 16, 32, 32) unchanged in shape
        x = self.pool(x)    # (B, 16, 32, 32) -> (B, 16, 16, 16)  (halved)

        # --- Block 2: conv2 + ReLU + MaxPool ---
        x = self.conv2(x)   # (B, 16, 16, 16) -> (B, 32, 16, 16)
        x = F.relu(x)       # (B, 32, 16, 16) unchanged in shape
        x = self.pool(x)    # (B, 32, 16, 16) -> (B, 32, 8, 8)   (halved again)

        # --- Flatten ---
        # Reshape from (B, 32, 8, 8) to (B, 2048) for the fully connected layers
        x = self.flatten(x) # (B, 32, 8, 8) -> (B, 2048)

        # --- Fully connected block ---
        x = self.fc1(x)     # (B, 2048) -> (B, 128)
        x = F.relu(x)       # non-linearity
        x = self.fc2(x)     # (B, 128)  -> (B, 64)
        x = F.relu(x)       # non-linearity
        x = self.fc3(x)     # (B, 64)   -> (B, 100): raw logit scores for each class

        # Return raw logits (unnormalized scores).
        # CrossEntropyLoss internally applies Softmax, so we do NOT apply it here.
        return x

# ---------------------------------------------------------------------------
# Training setup
# ---------------------------------------------------------------------------

# Instantiate the network
net = Net()

# Loss function: CrossEntropyLoss
# Combines LogSoftmax + NLLLoss in one step.
# It measures how different the predicted class distribution is from the true label.
# Lower loss = predictions are more confident and correct.
criterion = nn.CrossEntropyLoss()

# Optimizer: Stochastic Gradient Descent (SGD)
# Updates network parameters in the direction that reduces the loss.
# lr (learning rate) = 0.001 : controls how large each parameter update step is.
#   Too large -> unstable training; too small -> very slow convergence.
# momentum = 0.9 : accumulates a fraction of past gradients to smooth updates
#   and accelerate convergence, especially in flat or noisy loss landscapes.
optimizer = optim.SGD(net.parameters(), lr=0.001, momentum=0.9)

# ---------------------------------------------------------------------------
# Training loop
# ---------------------------------------------------------------------------

# Switch the network to training mode.
# This enables layers that behave differently during training vs. evaluation
# (e.g. Dropout randomly drops neurons; BatchNorm uses batch statistics).
net.train()

t0 = time.perf_counter()  # record the start time for measuring elapsed time

for epoch in range(epochs):  # loop over the dataset 'epochs' times

    running_loss = 0.0  # accumulates the loss over the reporting interval

    for i, data in enumerate(trainloader):  # iterate over mini-batches
        # data is a list [inputs, labels]
        # inputs : tensor of shape (batch_size, 3, 32, 32) — the image pixels
        # labels : tensor of shape (batch_size,) — the ground-truth class indices (0-99)
        inputs, labels = data

        # Zero out gradients from the previous iteration.
        # PyTorch accumulates gradients by default; we must reset them each step
        # to avoid adding gradients from different mini-batches together.
        optimizer.zero_grad()

        # Forward pass: feed inputs through the network to get predictions.
        # outputs shape: (batch_size, 100) — one logit per class per image
        outputs = net(inputs)

        # Compute the loss between predictions and ground-truth labels.
        loss = criterion(outputs, labels)

        # Backward pass: compute gradients of the loss with respect to all
        # learnable parameters using backpropagation (chain rule).
        loss.backward()

        # Parameter update: adjust each weight/bias by a small step
        # in the direction that reduces the loss (gradient descent).
        optimizer.step()

        # Accumulate the scalar loss value for logging.
        # .item() extracts a plain Python float from the 0-dim tensor.
        running_loss += loss.item()

        # Print a progress report every 2000 mini-batches.
        if i % 2000 == 1999:
            t1 = time.perf_counter()
            print('[%d, %5d] loss: %.3f, time: %.3f' %
                  (epoch + 1, i + 1, running_loss / 2000, t1 - t0))
            running_loss = 0.0  # reset the accumulator for the next interval
            t0 = t1             # reset the timer

print('Finished Training')

# ---------------------------------------------------------------------------
# Evaluation
# ---------------------------------------------------------------------------

# Switch to evaluation mode.
# Disables Dropout and makes BatchNorm use running statistics instead of batch statistics,
# ensuring deterministic and consistent predictions at test time.
net.eval()

correct = 0  # number of correctly classified test images
total = 0    # total number of test images processed

# torch.no_grad() disables gradient computation during evaluation.
# This saves memory and speeds up inference since we do not need gradients here.
with torch.no_grad():
    for data in testloader:
        images, labels = data

        # Forward pass: get logit scores for each class
        outputs = net(images)  # shape: (100, 100) — 100 images x 100 classes

        # torch.max returns the maximum value and its index along a dimension.
        # dim=1 searches across the 100 class scores for each image.
        # '_' discards the max value; 'predicted' holds the predicted class index.
        _, predicted = torch.max(outputs.data, 1)  # shape: (100,)

        # Count how many images were in this batch and add to the running total
        total += labels.size(0)

        # Count how many predictions match the ground-truth labels in this batch
        correct += (predicted == labels).sum().item()

# Print the final test accuracy as a percentage.
# With only 2 epochs and this small network, accuracy will be low (~5-10%).
# This notebook is a baseline — later notebooks will improve upon it.
print('Accuracy of the network on the 10000 test images: %.3f %%' % (100 * correct / total))


In [5]:
# Cell 3: Execute a Python file
%cd "{PROJECT_PATH}"
!python 00first.py

/content/drive/MyDrive/Practical-ML/cifar100
100% 169M/169M [40:25<00:00, 69.7kB/s]
[1,  2000] loss: 4.592, time: 19.606
[1,  4000] loss: 4.402, time: 18.612
[1,  6000] loss: 4.080, time: 19.145
[1,  8000] loss: 3.921, time: 18.772
[1, 10000] loss: 3.811, time: 18.371
[1, 12000] loss: 3.714, time: 18.486
[2,  2000] loss: 3.544, time: 24.520
[2,  4000] loss: 3.464, time: 18.700
[2,  6000] loss: 3.375, time: 19.999
[2,  8000] loss: 3.332, time: 18.935
[2, 10000] loss: 3.250, time: 19.087
[2, 12000] loss: 3.178, time: 19.679
Finished Training
Accuracy of the network on the 10000 test images: 21.660 %


💾 **Don't forget to save this notebook!**

To keep your work, go to File → Save a copy in Drive before closing.

[![Return to GitHub](https://img.shields.io/badge/GitHub-Return-black?logo=github)](https://github.com/mastnk/practical-ml)